In [ ]:
import torch
import torchtext
import numpy as np
import torchtext
import pandas
import collections
import collections
from torchnlp import *
from torch.optim import lr_scheduler

# ???????? Simple RNN ? ???????????? ? ??-??????? ????????????

### ??????????????

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# ???????????? ?? ??????
tokenizer = torchtext.data.utils.get_tokenizer('basic_english')
# ????? ?????????????? ?????????????????? ???????
nchars = 100
num_layers=3
test_str_orig = "??????????? ??????? ??????????? ? ?????? ?????? ?? ????. ??????? ??????? ? ??????? ?? ??????? ??????? ? ???? ?? ?????. ???? ?????? ???????? ??????, ??????? ?????, ????? ??????? ??? ?????? ????????, ?? ???-???? ? ??????????? ?????????? ??????."
test_str_slov = "??????????? ??????? ??????????? ? ??????"
test_str = test_str_orig[:nchars]


## ???????? ???????? ? ???????????? ?? ??????

In [ ]:
def load_dataset_HP(ngrams=1,min_freq=1):
    global vocab, tokenizer
    print("Loading dataset...")
    train_dataset = pandas.read_csv('./dataHPMRM.csv')["text"]
    train_dataset = list(train_dataset)
    print('Building vocab...')
    counter = collections.Counter()
    for line in train_dataset:
        counter.update(torchtext.data.utils.ngrams_iterator(tokenizer(line),ngrams=ngrams))
    vocab = torchtext.vocab.vocab(counter, min_freq=min_freq)
    return train_dataset,vocab
train_dataset,vocab_slov=load_dataset_HP()
vocab_size_slov = len(vocab_slov)

### ???????????? ???????????

In [ ]:
def char_tokenizer(words):
    return list(words) #[word for word in words]

counter = collections.Counter()
for line in train_dataset:
    counter.update(char_tokenizer(line))
vocab = torchtext.vocab.vocab(counter)

vocab_size = len(vocab)
print(f"Vocabulary size = {vocab_size}")
print(f"Encoding of 'a' is {vocab.get_stoi()['a']}")
print(f"Character with code 13 is {vocab.get_itos()[13]}")


### ??????? ??????????? ???????

In [ ]:
stoi_hash = {}
def encode(x,voc=None,unk=0,tokenizer=tokenizer):
    global stoi_hash
    v = vocab if voc is None else voc
    if v in stoi_hash.keys():
        stoi = stoi_hash[v]
    else:
        stoi = v.get_stoi()
        stoi_hash[v]=stoi        
    return [stoi.get(s,unk) for s in tokenizer(x)]

In [ ]:
## ?-?? ??????????? ??????? ?? ??????
def enc_slov(x):
    return torch.LongTensor(encode(x,voc=vocab_slov,tokenizer=tokenizer))

In [ ]:
## ?-?? ??????????? ??????? ?? ????????
def enc(x):
    return torch.LongTensor(encode(x,voc=vocab,tokenizer=char_tokenizer))

### ??????? ???????? ????? ? ?????

In [ ]:
def get_batch(s,nchars=nchars):
    ins = torch.zeros(len(s)-nchars,nchars,dtype=torch.long,device=device)
    outs = torch.zeros(len(s)-nchars,nchars,dtype=torch.long,device=device)
    for i in range(len(s)-nchars):
        ins[i] = enc(s[i:i+nchars])
        outs[i] = enc(s[i+1:i+nchars+1])
    return ins,outs

In [ ]:
def get_batch_slov(s,nchars=nchars):
    leng = len(enc_slov(s))
    inps = torch.zeros(leng-nchars,nchars,dtype=torch.long,device=device)
    outs = torch.zeros(leng-nchars,nchars,dtype=torch.long,device=device)
    for i in range(leng-nchars):
        inps[i] = enc_slov(s)[i:i+nchars]
        outs[i] = enc_slov(s)[i+1:i+nchars+1]
    return inps,outs

## ??????????? ??????????????? ???? Simple RNN

In [ ]:
class RNNGenerator(torch.nn.Module):
    def __init__(self, vocab_size, hidden_dim):
        super().__init__()
        self.vocab_size=vocab_size
        self.rnn = torch.nn.RNN(self.vocab_size,hidden_dim,batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, self.vocab_size)

    def forward(self, x):
        x = torch.nn.functional.one_hot(x,self.vocab_size).to(torch.float32)
        x,h = self.rnn(x)
        return self.fc(x)

## ????????? ?????????? ????????????????? ????????/????

In [ ]:
def generate(net,size=100,start='??????? '):
    chars = list(start)
    out = net(enc(chars).view(1,-1).to(device))
    for i in range(size):
        nc = torch.argmax(out[0][-1])
        chars.append(vocab.get_itos()[nc])
        out = net(nc.view(1,-1))
    return ''.join(chars)
        
def generate_slov(net,size=100,start='??????? '):
    chars = start
    out = net(enc_slov(start).view(1,-1).to(device))
    for i in range(size):
        nc = torch.argmax(out[0][-1])
        chars+= " " + (vocab_slov.get_itos()[nc])
        out = net(nc.view(1,-1))
    return ''.join(chars)

## ?????? ?????????

In [ ]:
def generate_soft(net,size=100,start='??????? ??? ',temperature=1.0,enc=enc):
    chars = list(start)
    out = net(enc(chars).view(1,-1).to(device))
    for i in range(size):
        #nc = torch.argmax(out[0][-1])
        out_dist = out[0][-1].div(temperature).exp()
        nc = torch.multinomial(out_dist,1)[0]
        chars.append(vocab.get_itos()[nc])
        out = net(nc.view(1,-1))
    return ''.join(chars)

def generate_soft_slov(net,size=100,start='??????? ??? ',temperature=1.0,enc=enc):
    chars = start
    out = net(enc_slov(chars).view(1,-1).to(device))
    for i in range(size):
        #nc = torch.argmax(out[0][-1])
        out_dist = out[0][-1].div(temperature).exp()
        nc = torch.multinomial(out_dist,1)[0]
        chars += " " + vocab_slov.get_itos()[nc]
        out = net(nc.view(1,-1))
    return ''.join(chars)


## ?-?? ????????

In [ ]:
def train(net,train_dataset,val_dataset,num_epochs, optimizer, scheduler, get_batch, generate, slov = False):
    for epoch in range(num_epochs):
        print(f"Epoch {epoch}/{num_epochs}")
        for phase in ["T","V"]:
            if phase == 'T':
                net.train()  # ?????????? ?????? ? ????? ????????
                dataset=train_dataset
            else:
                net.eval()   #?????????? ?????? ? ????? ??????
                dataset=val_dataset
            
            epoch_loss = 0.0    
            for i,x in enumerate(dataset):
                
                # x[0] is class label, x is text
                if slov:
                    leng = len(enc_slov(x))
                else:
                    leng = len(x)
                    
                if leng-nchars<10:
                    continue

                text_in, text_out = get_batch(x,nchars=nchars)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'T'):
                    out = net(text_in)
                    loss = torch.nn.functional.cross_entropy(out.view(-1,vocab_size),text_out.flatten()) #cross_entropy(out,labels)
                    if phase == 'T':
                        loss.backward()
                        optimizer.step()
                    else:
                        # scheduler.step(loss.item())
                        pass
                epoch_loss+=loss.item()
                # if i%100==0:
                #     print(f"{i}/{len(dataset)} Current loss {phase} = {epoch_loss/(i+1)}")
            print(f"Current loss {phase} = {epoch_loss / len(dataset)}")
            if phase == 'V':
                print(generate(net,start=test_str))    

## ???????? Simple RNN ? ???????????? ????????????

In [ ]:
net=RNNGenerator(vocab_size,nchars).to(device)
optimizer = torch.optim.Adam(net.parameters(),0.01)
scheduler=lr_scheduler.ReduceLROnPlateau(optimizer, 'min',factor=0.9,patience=50)

In [ ]:
train(net,train_dataset,train_dataset,10,optimizer, scheduler, get_batch, generate)

In [ ]:
for i in [0.3,0.8,1.0,1.3,1.8]:
    print(f"--- Temperature = {i}\n{generate_soft(net,size=300,start=test_str_slov,temperature=i ,enc=enc)}\n")

## ???????? Simple RNN ? ????????? ????????????

In [ ]:
nchars = 10
vocab_size = len(vocab_slov)

In [ ]:
net=RNNGenerator(vocab_size,nchars).to(device)
optimizer = torch.optim.Adam(net.parameters(),0.01)
scheduler=lr_scheduler.ReduceLROnPlateau(optimizer, 'min',factor=0.9,patience=50)

In [ ]:
train(net,train_dataset,train_dataset,10,optimizer, scheduler, get_batch_slov, generate_slov, slov = True)

In [ ]:
print(f"?????????\n{generate_slov(net,size=300,start=test_str_slov)}\n")